# DELTA1 v2.8 production-candidate review

> **PRODUCTION STATUS: BLOCKED — NOT PRODUCTION READY**

This notebook is a thin, read-only client over `strategy.py` and
`production.py`. It runs the canonical historical pipeline once, then uses
the returned ledgers and public reporting functions for performance,
trade-episode, uncertainty, risk-control, and readiness diagnostics. It
does not reproduce signal, sizing, execution, or cost logic.

The strategy was selected retrospectively, every reported period has been
reused, and the available futures history ends in 2014. Results are
research evidence only; they are not an independent forecast of live
performance.

## 1. Reproducible setup

Set `DELTA1_DATA_DIR` to the supplied Delta1 directory. If it is absent,
the notebook uses the repository-local
`Round1AllData/Quant Researcher/Delta1` path. The canonical ledger starts
on 1990-01-01 with $1 million and zero positions; earlier observations are
warm-up data only.

In [1]:
import os
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import diagnostics
import production as controls
import strategy as runtime
import stress

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

candidates = []
configured_data_dir = os.environ.get("DELTA1_DATA_DIR")
if configured_data_dir:
    candidates.append(Path(configured_data_dir).expanduser())
candidates.append(Path("Round1AllData/Quant Researcher/Delta1"))
data_dir = next((path.resolve() for path in candidates if path.is_dir()), None)
if data_dir is None:
    searched = "; ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "DELTA1 data not found. Set DELTA1_DATA_DIR. Searched: " + searched
    )

config = runtime.StrategyConfig(data_dir=data_dir, output_dir=Path("outputs"))
# Keep independent live gates distinct from the tighter research intent
# buffers in StrategyConfig (6x/35% live versus 5x/30% research).
production_limits = controls.ProductionLimits()

print(f"Strategy: {runtime.STRATEGY_NAME}")
print(f"Engine:   {runtime.ENGINE_VERSION}")
print("Data:     ${DELTA1_DATA_DIR} (resolved locally)")
print(f"Launch:   {config.launch_date}; ${config.initial_capital:,.0f}; zero positions")
print(f"Mode:     {config.mode} (production mode is intentionally fail-closed)")

Strategy: Production-Candidate Global TSMOM + Basis Momentum
Engine:   2.8.0
Data:     ${DELTA1_DATA_DIR} (resolved locally)
Launch:   1990-01-01; $1,000,000; zero positions
Mode:     research (production mode is intentionally fail-closed)


## 2. Canonical execution, friction, and control contract

A month-end decision fixes contract quantity using decision-date NAV. The
canonical order fills at the next eligible positive-volume close after any
configured additional delay, and new contracts earn P&L only after that
fill. The ledger uses integer contracts, charges ordinary and two-leg roll
turnover, and applies gross-notional and static-margin intent buffers before
and after rounding.

Frictions are parameterized research estimates—not calibration evidence.
Roll capacity remains unverified because continuous data do not provide
serial-expiry order books.

In [2]:
execution_contract = pd.Series(
    {
        "Execution timing": config.execution_timing,
        "Additional eligible-session delay": config.execution_delay_sessions,
        "Half-spread (ticks)": config.half_spread_ticks,
        "Additional slippage (ticks)": config.slippage_ticks,
        "Commission / one-way contract (USD)": config.commission_per_contract,
        "Exchange + regulatory fees / contract (USD)": (
            config.exchange_and_regulatory_fees_per_contract
        ),
        "Impact at full participation (bps)": config.impact_bps_at_full_participation,
        "Rebalance participation cap": config.max_rebalance_participation,
        "Research gross-notional intent buffer": config.max_gross_notional_multiple,
        "Research static-margin intent buffer": config.max_static_margin_fraction,
        "Charge roll costs": config.charge_roll_costs,
    },
    name="Canonical v2.8 assumption",
)
display(execution_contract.to_frame())

,Canonical v2.8 assumption
Execution timing,next_close
Additional eligible-session delay,0
Half-spread (ticks),0.5
Additional slippage (ticks),0.25
Commission / one-way contract (USD),2.5
Exchange + regulatory fees / contract (USD),1.5
Impact at full participation (bps),10.0
Rebalance participation cap,0.02
Research gross-notional intent buffer,5.0
Research static-margin intent buffer,0.3


## 3. Canonical run and launch-boundary checks

This is the notebook's only canonical pipeline invocation. Subsequent
sections consume `result` and `metrics`. Optional runtime stress/reporting
APIs may perform their own explicitly labeled scenario work.

In [3]:
result, metrics = runtime.run_pipeline(config)

launch = pd.Timestamp(config.launch_date)
pre_launch = result.daily.index < launch
live_dates = result.daily.index[result.daily.index >= launch]
if live_dates.empty:
    raise RuntimeError("The backtest contains no ledger rows on or after launch")
first_live_date = live_dates[0]
launch_audit = pd.Series(
    {
        "Configured launch": config.launch_date,
        "First live ledger date": first_live_date.date().isoformat(),
        "Opening NAV on first live date (USD)": result.daily.loc[
            first_live_date, "prior_nav_usd"
        ],
        "End-of-date contracts after any eligible launch fill": (
            result.positions.loc[first_live_date].abs().sum()
        ),
        "Pre-launch NAV fixed at initial capital": bool(
            np.allclose(result.daily.loc[pre_launch, "nav"], config.initial_capital)
        ),
        "Pre-launch positions all zero": bool(
            result.positions.loc[pre_launch].eq(0).all().all()
        ),
        "Reporting status": ", ".join(sorted(metrics["Validation status"].unique())),
    },
    name="Launch and provenance audit",
)
display(launch_audit.to_frame())

,Launch and provenance audit
Configured launch,1990-01-01
First live ledger date,1990-01-01
Opening NAV on first live date (USD),1000000.0
End-of-date contracts after any eligible launch fill,0.0
Pre-launch NAV fixed at initial capital,True
Pre-launch positions all zero,True
Reporting status,retrospective_reused_history


## 4. Extended metrics, NAV, drawdown, and costs

The full post-launch history is the headline view. Development and reused
later slices are shown only to diagnose stability. No row is an independent
holdout and no performance threshold is applied.

In [4]:
metric_columns = [
    "Window",
    "Validation status",
    "CAGR",
    "Annualized volatility",
    "Monthly Sharpe (rf=0)",
    "HAC Sharpe (21 lags, rf=0)",
    "Sortino (rf=0)",
    "Historical daily CVaR 95%",
    "Max drawdown",
    "Max drawdown duration (sessions)",
    "Annual cost drag",
    "Annual fixed-cost drag",
    "Annual impact-cost drag",
    "Peak gross notional multiple",
    "Peak static margin fraction",
    "Peak order participation",
    "Peak rebalance participation",
    "Peak roll participation proxy",
    "Closed trade episodes",
    "Trade profit factor (contribution)",
    "Trade expectancy (bps NAV)",
]
metric_view = metrics[[column for column in metric_columns if column in metrics]].set_index("Window")
percent_columns = {
    "CAGR",
    "Annualized volatility",
    "Historical daily CVaR 95%",
    "Max drawdown",
    "Annual cost drag",
    "Annual fixed-cost drag",
    "Annual impact-cost drag",
    "Peak static margin fraction",
    "Peak order participation",
    "Peak rebalance participation",
    "Peak roll participation proxy",
}
formats = {
    column: ("{:.2%}" if column in percent_columns else "{:.3f}")
    for column in metric_view.select_dtypes(include=[np.number]).columns
}
display(metric_view.style.format(formats))

,Validation status,CAGR,Annualized volatility,Monthly Sharpe (rf=0),"HAC Sharpe (21 lags, rf=0)",Sortino (rf=0),Historical daily CVaR 95%,Max drawdown,Max drawdown duration (sessions),Annual cost drag,Annual fixed-cost drag,Annual impact-cost drag,Peak gross notional multiple,Peak static margin fraction,Peak order participation,Peak rebalance participation,Peak roll participation proxy,Closed trade episodes,Trade profit factor (contribution),Trade expectancy (bps NAV)
Window,,,,,,,,,,,,,,,,,,,,
1990-2004 development history,retrospective_reused_history,16.26%,8.70%,1.674,1.603,2.569,1.23%,-10.49%,242.000,1.48%,1.34%,0.14%,5.271,32.23%,100.00%,1.99%,100.00%,920.000,2.366,20.984
2005-2014 reused later diagnostic,retrospective_reused_history,11.71%,10.29%,1.049,1.027,1.584,1.50%,-19.16%,533.000,0.91%,0.77%,0.14%,5.505,30.96%,4.45%,1.99%,4.45%,881.000,1.972,12.354
1990-2014 full post-launch history,retrospective_reused_history,14.42%,9.37%,1.391,1.345,2.117,1.34%,-19.16%,533.000,1.25%,1.11%,0.14%,5.505,32.23%,100.00%,1.99%,100.00%,1801.000,2.192,16.762


In [5]:
full_start, full_end = runtime.REPORTING_WINDOWS[
    "1990-2014 full post-launch history"
]
post_launch = result.daily.loc[full_start:full_end].copy()
equity = post_launch["equity"]
running_peak = np.maximum.accumulate(np.r_[1.0, equity.to_numpy()])[1:]
drawdown = pd.Series(equity.to_numpy() / running_peak - 1, index=equity.index)

annual_costs = post_launch[
    ["fixed_execution_cost_usd", "market_impact_cost_usd"]
].resample("YE").sum()
annual_costs.columns = ["Fixed execution", "Market impact"]

fig, axes = plt.subplots(
    3, 1, figsize=(12, 9), sharex=False,
    gridspec_kw={"height_ratios": [2.0, 1.0, 1.2]},
)
axes[0].plot(equity.index, equity, color="#225EA8", linewidth=1.4)
axes[0].set_yscale("log")
axes[0].set_ylabel("NAV / initial NAV")
axes[0].set_title("Canonical post-launch net equity (log scale)")
axes[1].fill_between(drawdown.index, drawdown * 100, 0, color="#CB181D", alpha=0.55)
axes[1].set_ylabel("Drawdown (%)")
annual_costs.plot.bar(stacked=True, ax=axes[2], color=["#3182BD", "#FDAE6B"])
axes[2].set_xticklabels(annual_costs.index.year, rotation=45, ha="right")
axes[2].set_ylabel("Annual cost (USD)")
axes[2].set_xlabel("Year")
axes[2].legend(frameon=False)
fig.tight_layout()
plt.show()

/var/folders/39/8k2n5j154v72pxxr9yybgbth0000gn/T/ipykernel_18969/3219963135.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Trade episodes and outcome concentration

A same-sign resize remains in one episode; a sign flip closes one and opens
another; rolls add cost without creating a new alpha episode. Open episodes
are censored. Episode counts are not independent observations because
markets overlap and share portfolio risk.

In [6]:
episodes = result.trade_episodes.copy()
trade_report = diagnostics.trade_metrics_report(
    result,
    runtime.REPORTING_WINDOWS,
)
episode_summary = runtime.trade_episode_metrics(episodes, full_start, full_end)
display(episode_summary.to_frame("Full post-launch history"))
display(Markdown("**Diagnostic episode report: overall rows**"))
display(trade_report.loc[trade_report["Scope"].eq("Overall")])
display(Markdown("**Diagnostic episode report: full-history asset classes**"))
display(
    trade_report.loc[
        trade_report["Window"].eq("1990-2014 full post-launch history")
        & trade_report["Scope"].eq("Asset class")
    ]
)

closed = episodes.loc[episodes["Status"].eq("Closed")].copy()
if closed.empty:
    display(Markdown("**No closed trade episodes are available.**"))
else:
    asset_episode_summary = closed.groupby("Asset class").agg(
        Episodes=("Episode ID", "count"),
        Net_PnL_USD=("Net P&L USD", "sum"),
        Net_contribution=("Net contribution", "sum"),
        Median_holding_sessions=("Holding sessions", "median"),
        Roll_count=("Roll count", "sum"),
    ).sort_values("Net_contribution", ascending=False)
    display(
        asset_episode_summary.style.format({
            "Net_PnL_USD": "${:,.0f}",
            "Net_contribution": "{:.2%}",
            "Median_holding_sessions": "{:.0f}",
        })
    )

    episode_columns = [
        "Episode ID", "Symbol", "Asset class", "Direction", "Entry date",
        "Exit date", "Holding sessions", "Net P&L USD", "Net contribution",
        "MFE contribution", "MAE contribution", "Resize count", "Roll count",
    ]
    display(Markdown("**Largest positive closed episodes**"))
    display(closed.nlargest(10, "Net contribution")[episode_columns])
    display(Markdown("**Largest negative closed episodes**"))
    display(closed.nsmallest(10, "Net contribution")[episode_columns])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    closed["Holding sessions"].clip(upper=closed["Holding sessions"].quantile(0.99)).hist(
        bins=35, ax=axes[0], color="#756BB1"
    )
    axes[0].set_title("Closed-episode holding sessions (trimmed at 99th percentile)")
    axes[0].set_xlabel("Sessions")
    asset_episode_summary["Net_contribution"].sort_values().plot.barh(
        ax=axes[1], color="#31A354"
    )
    axes[1].set_title("Net episode contribution by asset class")
    axes[1].set_xlabel("Sum of episode contributions")
    fig.tight_layout()
    plt.show()

,Full post-launch history
Closed trade episodes,1801.000000
Open/censored trade episodes,52.000000
Episodes entered before window,0.000000
Winning trade episodes,884.000000
Losing trade episodes,917.000000
Breakeven trade episodes,0.000000
Trade win rate,0.490838
Non-breakeven trade win rate,0.490838
Trade profit factor (contribution),2.191789
Trade profit factor (USD),1.737791


**Diagnostic episode report: overall rows**

,Window,Cohort definition,Scope,Group,Closed episodes,Open or censored episodes,Entries before window,Wins,Losses,Breakevens,Win rate,Loss rate,Non-breakeven win rate,Profit factor contribution,Profit factor USD,Profit factor denominator zero,Expectancy bps,Expectancy USD,Average win bps,Average loss bps,Payoff ratio contribution,Average win USD,Average loss USD,Payoff ratio USD,Median outcome bps,Mean holding sessions,Median holding sessions,Low sample,Definition version
0,1990-2004 development history,closed episodes grouped by exit date,Overall,All,920,47,0,457,463,0,0.496739,0.503261,0.496739,2.366287,2.187553,False,20.983944,5894.182980,73.161765,30.517707,2.397355,21857.514916,9862.280723,2.216274,0.108693,138.015217,65.0,False,directional episode v1
68,2005-2014 reused later diagnostic,closed episodes grouped by exit date,Overall,All,881,52,47,427,454,0,0.484677,0.515323,0.484677,1.971662,1.628222,False,12.353579,13365.770955,51.719944,24.671614,2.096334,71473.055358,41285.793891,1.731178,-0.598726,139.057889,65.0,False,directional episode v1
136,1990-2014 full post-launch history,closed episodes grouped by exit date,Overall,All,1801,52,0,884,917,0,0.490838,0.509162,0.490838,2.191789,1.737791,False,16.762205,9549.079707,62.804686,27.623349,2.273609,45823.392482,25419.832499,1.802663,-0.392095,138.525264,65.0,False,directional episode v1


**Diagnostic episode report: full-history asset classes**

,Window,Cohort definition,Scope,Group,Closed episodes,Open or censored episodes,Entries before window,Wins,Losses,Breakevens,Win rate,Loss rate,Non-breakeven win rate,Profit factor contribution,Profit factor USD,Profit factor denominator zero,Expectancy bps,Expectancy USD,Average win bps,Average loss bps,Payoff ratio contribution,Average win USD,Average loss USD,Payoff ratio USD,Median outcome bps,Mean holding sessions,Median holding sessions,Low sample,Definition version
137,1990-2014 full post-launch history,closed episodes grouped by exit date,Asset class,Agriculture & livestock,510,13,0,229,281,0,0.449020,0.550980,0.449020,1.822821,1.232108,False,13.411229,3245.938549,66.167052,29.581952,2.236737,38373.642187,25381.264772,1.511888,-2.225146,146.572549,66.0,False,directional episode v1
138,1990-2014 full post-launch history,closed episodes grouped by exit date,Asset class,Energy,194,3,0,104,90,0,0.536082,0.463918,0.536082,2.452902,1.634923,False,24.312031,9484.471600,76.565573,36.069840,2.122703,45557.280283,32199.662878,1.414837,3.237332,147.917526,66.5,False,directional episode v1
139,1990-2014 full post-launch history,closed episodes grouped by exit date,Asset class,Equity indices,320,14,0,169,151,0,0.528125,0.471875,0.528125,2.480859,2.086694,False,15.882756,13240.891046,50.382241,22.729250,2.216626,48142.876721,25821.596234,1.864442,1.267875,123.212500,64.0,False,directional episode v1
140,1990-2014 full post-launch history,closed episodes grouped by exit date,Asset class,FX,243,6,0,118,125,0,0.485597,0.514403,0.485597,2.013361,1.283958,False,14.803138,4623.169041,60.566919,28.397872,2.132798,43048.811352,31650.637301,1.360125,-0.896956,134.283951,45.0,False,directional episode v1
141,1990-2014 full post-launch history,closed episodes grouped by exit date,Asset class,Government bonds,369,12,0,186,183,0,0.504065,0.495935,0.504065,2.717921,2.782473,False,20.863437,17224.218459,65.483660,24.488266,2.674083,53340.976627,19484.617711,2.737594,0.070121,145.913279,66.0,False,directional episode v1
142,1990-2014 full post-launch history,closed episodes grouped by exit date,Asset class,Metals,165,4,0,78,87,0,0.472727,0.527273,0.472727,1.976447,2.068549,False,13.661930,12037.724723,58.497592,26.535560,2.204498,49295.258794,21365.581684,2.307228,-1.370950,122.030303,44.0,False,directional episode v1


...

**Largest positive closed episodes**

,Episode ID,Symbol,Asset class,Direction,Entry date,Exit date,Holding sessions,Net P&L USD,Net contribution,MFE contribution,MAE contribution,Resize count,Roll count
29,GF-0002,GF,Agriculture & livestock,Long,1990-04-02,1993-06-01,826,91747.500210,0.078631,0.078951,-0.000195,25,26
452,ZW-0012,ZW,Agriculture & livestock,Short,1997-05-01,2002-03-01,1261,181587.076177,0.056611,0.058845,-0.000017,44,24
870,HG-0029,HG,Metals,Long,2003-08-01,2006-05-01,716,574346.887062,0.055699,0.055706,-0.002301,25,14
60,YYT-0004,YYT,Government bonds,Long,1990-10-01,1994-05-02,935,60342.410969,0.055659,0.076780,-0.000212,34,14
749,6A-0009,6A,FX,Long,2002-03-01,2005-09-01,914,334844.698216,0.046313,0.050648,-0.000025,34,14
470,KE-0011,KE,Agriculture & livestock,Short,1997-10-01,2002-07-01,1238,146392.915780,0.043191,0.045536,-0.003582,47,24
75,ZF-0003,ZF,Government bonds,Long,1991-01-02,1994-04-01,847,50262.711689,0.042111,0.052971,-0.005683,29,13
325,ZC-0006,ZC,Agriculture & livestock,Long,1995-07-03,1997-04-01,456,82903.895892,0.041518,0.046231,-0.001084,15,8
486,ZC-0009,ZC,Agriculture & livestock,Short,1998-01-02,2003-03-03,1346,148579.704902,0.041069,0.047136,-0.003303,55,26
844,GF-0024,GF,Agriculture & livestock,Long,2003-04-01,2006-01-03,720,342771.199377,0.037908,0.038372,-0.000018,28,22


**Largest negative closed episodes**

,Episode ID,Symbol,Asset class,Direction,Entry date,Exit date,Holding sessions,Net P&L USD,Net contribution,MFE contribution,MAE contribution,Resize count,Roll count
964,NG-0026,NG,Energy,Short,2004-12-01,2005-09-01,196,-251589.491632,-0.024409,0.002078,-0.024409,6,9
160,6B-0009,6B,FX,Long,1992-09-01,1992-11-02,44,-27451.559590,-0.021275,0.000257,-0.021275,0,1
33,ZF-0002,ZF,Government bonds,Short,1990-05-01,1991-01-02,176,-21918.262087,-0.021166,0.000156,-0.021393,5,3
7,HG-0001,HG,Metals,Short,1990-01-02,1990-05-01,85,-16507.833736,-0.016868,0.002639,-0.017739,3,2
14,ZF-0001,ZF,Government bonds,Long,1990-01-02,1990-05-01,85,-15620.770575,-0.015818,0.000000,-0.015828,3,1
15,ZL-0001,ZL,Agriculture & livestock,Short,1990-01-02,1990-05-01,85,-15468.383141,-0.015729,0.000000,-0.016826,2,2
564,PL-0021,PL,Metals,Short,1999-07-01,1999-10-01,66,-54010.206831,-0.015653,0.001883,-0.020424,2,1
354,RS-0008,RS,Agriculture & livestock,Short,1995-12-01,1996-05-01,108,-31113.776429,-0.015209,0.000557,-0.015209,4,2
97,NG-0001,NG,Energy,Short,1991-07-01,1992-06-01,240,-16987.440373,-0.014935,0.001730,-0.014935,6,11
25,CT-0001,CT,Agriculture & livestock,Short,1990-03-01,1990-06-01,66,-14340.739383,-0.014662,0.000000,-0.015688,1,1


/var/folders/39/8k2n5j154v72pxxr9yybgbth0000gn/T/ipykernel_18969/3917902250.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Friction and capacity stress

`stress.py` reports the canonical assumptions and reruns seven predefined
retrospective scenarios. Each scenario uses the strategy runtime and is
labeled as reused-history sensitivity, not validation.

In [7]:
cost_assumptions = stress.cost_model_assumptions(config)
display(cost_assumptions)

friction_stress = stress.run_friction_stress_suite(config)
stress_view = friction_stress.set_index("scenario")
display(
    stress_view.style.format({
        "cagr": "{:.2%}",
        "annualized_volatility": "{:.2%}",
        "sharpe": "{:.3f}",
        "sortino": "{:.3f}",
        "max_drawdown": "{:.2%}",
        "annual_cost_drag": "{:.2%}",
        "annual_fixed_cost_drag": "{:.2%}",
        "annual_impact_cost_drag": "{:.2%}",
        "peak_static_margin_fraction": "{:.2%}",
        "peak_order_participation": "{:.2%}",
    })
)

,parameter,value,category,unit,description
0,half_spread_ticks,0.5,fixed,ticks per contract per side,Half of the quoted bid/ask spread charged on e...
1,slippage_ticks,0.25,fixed,ticks per contract per side,Additional adverse movement charged on each ex...
2,commission_per_contract,2.5,fixed,USD per contract per side,Broker commission charged on each execution.
3,exchange_and_regulatory_fees_per_contract,1.5,fixed,USD per contract per side,Exchange and regulatory fees charged on each e...
4,impact_bps_at_full_participation,10.0,impact,bps of traded notional,Square-root market-impact coefficient at 100% ...
5,charge_roll_costs,True,rolls,boolean,Whether approximate two-leg continuous-contrac...
6,execution_timing,next_close,scheduling,execution convention,Observed price field used for delayed execution.
7,execution_delay_sessions,0,scheduling,sessions,Additional executable sessions waited after th...
8,max_rebalance_participation,0.02,capacity,fraction of session volume,Maximum fraction of observed session volume av...


,assumption,validation_status,start,end,cagr,annualized_volatility,sharpe,sortino,max_drawdown,episode_profit_factor,episode_profit_factor_usd,episode_expectancy_bps_nav,episode_expectancy_usd,annual_cost_drag,annual_fixed_cost_drag,annual_impact_cost_drag,peak_gross_notional_multiple,peak_static_margin_fraction,peak_order_participation
scenario,,,,,,,,,,,,,,,,,,,
baseline,Canonical configuration; no stress applied.,retrospective_sensitivity,1990-01-01,2014-12-31,14.42%,9.37%,1.436,2.117,-19.16%,2.191789,1.737791,16.762205,9549.079707,1.25%,1.11%,0.14%,5.504731,32.23%,100.00%
double_fixed_costs,"Half-spread, slippage, commission, and exchange/regulatory fees doubled.",retrospective_sensitivity,1990-01-01,2014-12-31,12.75%,9.34%,1.288,1.878,-19.92%,1.990007,1.648007,14.454589,6351.373519,2.38%,2.25%,0.13%,5.474744,31.83%,100.00%
double_impact,Market-impact coefficient doubled; fixed costs unchanged.,retrospective_sensitivity,1990-01-01,2014-12-31,14.14%,9.38%,1.410,2.073,-18.39%,2.176689,1.725344,16.240662,8791.283904,1.38%,1.11%,0.27%,5.507090,32.08%,100.00%
double_all_execution_costs,All fixed execution-cost inputs and the market-impact coefficient doubled.,retrospective_sensitivity,1990-01-01,2014-12-31,12.70%,9.34%,1.284,1.873,-20.01%,1.994954,1.647795,14.362319,6293.284419,2.50%,2.24%,0.25%,5.460802,31.78%,100.00%
additional_execution_delay_1_session,One executable session added to the configured execution delay.,retrospective_sensitivity,1990-01-01,2014-12-31,14.27%,9.45%,1.411,2.073,-18.45%,2.157012,1.719566,16.369466,9043.998736,1.26%,1.12%,0.14%,5.377860,33.08%,100.00%
tighter_participation_1pct,Maximum rebalance participation tightened to 1% of session volume.,retrospective_sensitivity,1990-01-01,2014-12-31,14.26%,9.40%,1.417,2.089,-18.68%,2.174808,1.746099,16.412201,9093.055371,1.25%,1.12%,0.14%,5.529545,32.14%,100.00%
combined_adverse,"All execution costs doubled, one session added, and participation capped at 1%.",retrospective_sensitivity,1990-01-01,2014-12-31,12.09%,9.34%,1.228,1.786,-19.52%,1.936339,1.618869,13.541296,5322.905848,2.47%,2.23%,0.24%,5.498701,31.60%,100.00%


## 7. Regime diagnostics

`causal_regime_report` uses lagged expanding terciles for volatility, trend
strength, cross-market correlation, and liquidity based only on earlier
months. Conditional rows are descriptive; discontiguous observations are
not stitched into artificial path metrics.

In [8]:
regime_report = diagnostics.causal_regime_report(
    result,
    start=full_start,
    end=full_end,
    minimum_history_months=36,
)
display(regime_report)

if not regime_report.empty:
    regime_plot = regime_report.pivot(
        index="Regime",
        columns="State",
        values="Mean monthly return",
    ).reindex(columns=["Low", "Middle", "High"])
    ax = regime_plot.plot.bar(figsize=(11, 4.2), color=["#9ECAE1", "#6BAED6", "#2171B5"])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Mean monthly return")
    ax.set_xlabel("")
    ax.set_title("Conditional outcomes under causal lagged regimes")
    ax.legend(title="State", frameon=False)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

,Regime,State,Months,Fraction of labeled months,Mean monthly return,Median monthly return,Monthly volatility,Annualized conditional mean,Positive month rate,Worst month,Average monthly cost,Average gross notional multiple,Average static margin fraction,Average maximum order participation,First labeled month,Last labeled month,Threshold method,Minimum threshold history months
0,Lagged realized volatility,Low,49,0.194444,0.009075,0.009773,0.029055,0.108897,0.673469,-0.070664,0.001025,4.471357,0.186477,0.013357,1994-02-28,2014-12-31,lagged expanding terciles,36
1,Lagged realized volatility,Middle,75,0.297619,0.017952,0.016961,0.030499,0.215423,0.813333,-0.062800,0.001071,4.144219,0.225488,0.011729,1994-01-31,2014-01-31,lagged expanding terciles,36
2,Lagged realized volatility,High,128,0.507937,0.008993,0.009293,0.030939,0.107920,0.601562,-0.058855,0.001020,3.638639,0.217306,0.010691,1994-03-31,2013-06-30,lagged expanding terciles,36
3,Lagged forecast magnitude,Low,116,0.441065,0.007843,0.008488,0.029299,0.094111,0.612069,-0.070664,0.001074,3.776619,0.212063,0.010478,1993-02-28,2014-03-31,lagged expanding terciles,36
4,Lagged forecast magnitude,Middle,88,0.334601,0.013913,0.016219,0.026528,0.166954,0.738636,-0.045970,0.001019,3.968539,0.206430,0.012618,1993-04-30,2014-12-31,lagged expanding terciles,36
5,Lagged forecast magnitude,High,59,0.224335,0.017494,0.016961,0.036073,0.209932,0.762712,-0.053869,0.001034,4.104121,0.230613,0.010848,1993-05-31,2014-08-31,lagged expanding terciles,36
6,Lagged liquidity pressure,Low,52,0.197719,0.010697,0.008502,0.035550,0.128362,0.653846,-0.056988,0.000948,3.857611,0.190617,0.009176,1993-07-31,2014-09-30,lagged expanding terciles,36
7,Lagged liquidity pressure,Middle,93,0.353612,0.012613,0.012178,0.030179,0.151360,0.720430,-0.070664,0.001106,3.677093,0.216507,0.010884,1993-02-28,2013-07-31,lagged expanding terciles,36
8,Lagged liquidity pressure,High,118,0.448669,0.012178,0.013679,0.027912,0.146131,0.677966,-0.062800,0.001044,4.126246,0.223085,0.012513,1993-03-31,2014-12-31,lagged expanding terciles,36
9,Lagged cross-market correlation,Low,49,0.186312,0.013210,0.012792,0.022719,0.158514,0.795918,-0.070664,0.001103,4.000210,0.224056,0.011690,1993-02-28,2014-12-31,lagged expanding terciles,36


/var/folders/39/8k2n5j154v72pxxr9yybgbth0000gn/T/ipykernel_18969/1753702102.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Block-bootstrap Monte Carlo uncertainty

The deterministic stationary bootstrap resamples post-launch monthly
returns across 6-, 12-, and 24-month expected blocks and 1-, 3-, 5-, and
10-year horizons. It preserves some serial dependence but remains
conditional on this retrospectively selected specification.

In [9]:
bootstrap = diagnostics.monthly_stationary_bootstrap_summary(
    result,
    {"1990-2014 full post-launch history": (full_start, full_end)},
    samples=2_000,
    block_lengths=(6, 12, 24),
    horizons_months=(12, 36, 60, 120),
    seed=20_260_803,
)
probability_rows = bootstrap.loc[bootstrap["Statistic"].eq("Probability")]
percentile_rows = bootstrap.loc[
    bootstrap["Statistic"].isin(["P05", "Median", "P95"])
]
display(probability_rows)
display(percentile_rows)

,Window,Source start,Source end,Source months,Method,Expected block months,Horizon months,Samples,Seed,Metric,Statistic,Value,Selection adjusted
28,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,12,2000,20260803,Probability of terminal loss,Probability,0.0705,False
29,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,12,2000,20260803,Probability maximum drawdown exceeds 20%,Probability,0.0000,False
30,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,12,2000,20260803,Probability maximum drawdown exceeds 30%,Probability,0.0000,False
31,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,12,2000,20260803,Probability maximum drawdown exceeds 40%,Probability,0.0000,False
60,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,36,2000,20260803,Probability of terminal loss,Probability,0.0055,False
61,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,36,2000,20260803,Probability maximum drawdown exceeds 20%,Probability,0.0015,False
62,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,36,2000,20260803,Probability maximum drawdown exceeds 30%,Probability,0.0000,False
63,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,36,2000,20260803,Probability maximum drawdown exceeds 40%,Probability,0.0000,False
92,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,60,2000,20260803,Probability of terminal loss,Probability,0.0000,False
93,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,60,2000,20260803,Probability maximum drawdown exceeds 20%,Probability,0.0070,False


,Window,Source start,Source end,Source months,Method,Expected block months,Horizon months,Samples,Seed,Metric,Statistic,Value,Selection adjusted
1,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,12,2000,20260803,CAGR,P05,-0.018664,False
3,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,12,2000,20260803,CAGR,Median,0.149579,False
5,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,12,2000,20260803,CAGR,P95,0.323215,False
8,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,12,2000,20260803,Monthly Sharpe,P05,-0.143981,False
10,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,6,12,2000,20260803,Monthly Sharpe,Median,1.481314,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,24,120,2000,20260803,Maximum drawdown,Median,-0.110715,False
371,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,24,120,2000,20260803,Maximum drawdown,P95,-0.064728,False
374,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,24,120,2000,20260803,Worst 12-month return,P05,-0.128618,False
376,1990-2014 full post-launch history,1990-01-31,2014-12-31,300,stationary monthly bootstrap,24,120,2000,20260803,Worst 12-month return,Median,-0.063865,False


## 9. Historical hard-control usage

The research ledger limits gross notional and static margin while creating
targets. The independent control layer also checks participation,
drawdown, ledger integrity, and terminal pending orders. A historical pass
is necessary but cannot prove live readiness.

In [10]:
reconciliation = runtime.ledger_reconciliation_report(result)
display(Markdown("**Ledger reconciliation**"))
display(reconciliation)

evidence = controls.ReadinessEvidence()
readiness = controls.production_readiness_report(
    result,
    limits=production_limits,
    evidence=evidence,
)
historical_gates = readiness.loc[~readiness["category"].eq("external evidence")]
display(historical_gates)

fig, axes = plt.subplots(3, 1, figsize=(12, 7.5), sharex=True)
post_launch["gross_notional_multiple"].plot(ax=axes[0], color="#225EA8")
axes[0].axhline(production_limits.max_gross_notional_multiple, color="#CB181D", linestyle="--")
axes[0].set_ylabel("Gross / NAV")
post_launch["static_margin_fraction"].plot(ax=axes[1], color="#31A354")
axes[1].axhline(production_limits.max_margin_fraction, color="#CB181D", linestyle="--")
axes[1].set_ylabel("Margin / NAV")
post_launch[[
    "max_rebalance_participation",
    "max_roll_participation_proxy",
]].plot(ax=axes[2], color=["#756BB1", "#E6550D"])
axes[2].axhline(production_limits.max_order_participation, color="#CB181D", linestyle="--")
axes[2].set_ylabel("Participation")
axes[2].set_xlabel("Date")
axes[2].legend(["Rebalance", "Roll proxy", "Hard limit"], frameon=False)
fig.suptitle("Historical control usage; dashed lines are production limits")
fig.tight_layout()
plt.show()

**Ledger reconciliation**

,check,status,maximum_absolute_error,tolerance,detail
0,market_gross_pnl_to_portfolio,PASS,2.328306e-10,0.000001,Sum of market gross P&L equals the portfolio g...
1,market_costs_to_portfolio,PASS,5.456968e-12,0.000001,Regular plus roll costs reconcile to the portf...
2,fixed_plus_impact_costs,PASS,1.818989e-12,0.000001,Parameterized fixed and impact costs reconcile...
3,position_change_to_trades,PASS,0.000000e+00,0.000001,Recorded trades equal the change in contract p...
4,net_pnl_identity,PASS,0.000000e+00,0.000001,Net P&L equals gross P&L less all modeled costs.
5,nav_recursion,PASS,3.725290e-09,0.000001,Closing NAV equals prior NAV plus net P&L.
6,net_return_identity,PASS,0.000000e+00,0.000001,Net return is calculated on prior closing NAV.
7,episode_gross_pnl_to_market_ledger,PASS,3.725290e-09,0.000001,Closed and censored directional episodes inclu...
8,episode_costs_to_market_ledger,PASS,0.000000e+00,0.000001,Closed and censored directional episodes inclu...
9,episode_contribution_to_daily_return,PASS,1.332268e-15,0.000001,Episode net contributions reconcile to additiv...


,gate,category,critical,status,observed,limit,reason
0,backtest_daily_available,backtest,True,PASS,9683,> 0 rows,gate passed
1,required_daily_columns,backtest,True,PASS,all present,all required columns,gate passed
2,finite_positive_nav,ledger,True,PASS,953968.963726,> 0,gate passed
3,finite_net_returns,ledger,True,PASS,finite,all finite,gate passed
4,nonnegative_costs,ledger,True,PASS,0.0,>= 0,gate passed
5,historical_gross_notional,risk,True,PASS,5.504731,6.0,gate passed
6,historical_margin,risk,True,PASS,0.322318,0.35,gate passed
7,historical_order_participation,risk,True,BLOCKED,1.0,0.02,historical order participation exceeds the pro...
8,historical_drawdown,risk,True,BLOCKED,0.191552,0.15,historical drawdown exceeds the production lim...
9,terminal_pending_orders,execution,True,PASS,0.0,0,gate passed


/var/folders/39/8k2n5j154v72pxxr9yybgbth0000gn/T/ipykernel_18969/766196743.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Fail-closed readiness and operational health

External evidence defaults to absent. Live health also defaults to blocked
because this notebook supplies no timestamped feed, broker connection,
reconciliation, monitoring, or kill-switch evidence. `READY` is possible
only when every critical gate explicitly passes.

In [11]:
overall_status = controls.overall_readiness_status(readiness)
live_health = controls.evaluate_runtime_health(limits=production_limits)

display(Markdown(f"### Overall production readiness: **{overall_status}**"))
display(
    readiness.groupby(["category", "status"]).size().rename("Gate count").to_frame()
)
display(readiness.loc[readiness["status"].ne(controls.PASS)])
display(Markdown(f"### Default live-health status: **{live_health.status}**"))
display(pd.Series(asdict(live_health), name="Default live-health check").to_frame())

### Overall production readiness: **BLOCKED**

Gate count
category          status             
backtest          PASS              2
execution         PASS              1
external evidence BLOCKED          11
ledger            PASS              3
risk              BLOCKED           2
                  PASS              2

,gate,category,critical,status,observed,limit,reason
7,historical_order_participation,risk,True,BLOCKED,1.0,0.02,historical order participation exceeds the pro...
8,historical_drawdown,risk,True,BLOCKED,0.191552,0.15,historical drawdown exceeds the production lim...
10,serial_contracts,external evidence,True,BLOCKED,False,True,Tradeable serial-contract history has not been...
11,timestamped_live_data,external evidence,True,BLOCKED,False,True,Timestamped live market data has not been evid...
12,dated_contract_specs,external evidence,True,BLOCKED,False,True,Dated contract specifications has not been evi...
13,dated_margin_and_fees,external evidence,True,BLOCKED,False,True,Dated margin and fee schedule has not been evi...
14,calibrated_cost_model,external evidence,True,BLOCKED,False,True,Calibrated execution-cost model has not been e...
15,independent_holdout,external evidence,True,BLOCKED,False,True,Independent holdout evaluation has not been ev...
16,paper_trading,external evidence,True,BLOCKED,False,True,Paper-trading evidence has not been evidenced
17,broker_adapter,external evidence,True,BLOCKED,False,True,Broker adapter has not been evidenced


### Default live-health status: **BLOCKED**

,Default live-health check
healthy,False
status,BLOCKED
data_age_seconds,None
gross_notional_multiple,None
margin_fraction,None
drawdown_fraction,None
broker_connected,False
broker_reconciled,False
monitoring_healthy,False
kill_switch_ready,False


## 11. Reproduction and artifacts

The notebook does not overwrite `outputs/`. Generate canonical artifacts
with:

```bash
delta1-strategy --data-dir "$DELTA1_DATA_DIR" --output-dir outputs
python -m unittest discover -s tests -v
```

The CLI writes portfolio and market ledgers, metrics, monthly position
intents, execution events, trade episodes/metrics, friction stress, causal
regimes, stationary-bootstrap summaries, readiness and reconciliation
reports, data quality, cost assumptions, source hashes, normalized config,
and a run manifest. See `README.md` for the exact filenames. Detailed
episodes are also available as `result.trade_episodes`; readiness is
produced by `production_readiness_report`.

This package remains blocked pending serial-contract data, timestamped live
inputs, dated specifications/margins/fees, calibrated execution costs,
independent evaluation, paper trading, broker integration/reconciliation,
monitoring, and a tested kill switch.